# Grazioso Salvare Rescue Dog Dashboard
**Sean Singh | CS 340 Client/Server Development | Project Two**

This dashboard helps Grazioso Salvare staff find dogs in the Austin Animal Center outcomes data that fit one of three search-and-rescue profiles. A row of radio buttons filters the data by rescue type. An interactive table, a breed pie chart and a location map all redraw to match.

The notebook follows Model-View-Controller:
- **Model**: MongoDB (`aac.animals`), reached only through the Project One CRUD module (`CRUD_Python_Module.AnimalShelter`)
- **View**: the Dash layout (logo, filter controls, data table, charts)
- **Controller**: the Dash callbacks that turn a filter choice into a MongoDB query and push the results to every widget

In [2]:
# Jupyter version of Dash, set up the same way as the Module Six milestone
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

# Dash building blocks: core controls, HTML wrappers, and the data table
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

# Leaflet map for the geolocation chart
import dash_leaflet as dl

# Plotly Express draws the breed pie chart
import plotly.express as px

# base64 turns the logo PNG into text the browser can display inline
import base64

# pandas sits between the CRUD query results and the Dash widgets
import pandas as pd

# Project One CRUD module, the dashboard's only way into MongoDB
from CRUD_Python_Module import AnimalShelter

In [3]:
###########################
# Model: data through the CRUD module
###########################
# Restricted account created in Project One (readWrite on aac only)
username = "aacuser"
password = "PAWAN"

# The constructor pings MongoDB, so a wrong password or a stopped
# server fails here instead of halfway through the dashboard
db = AnimalShelter(username, password)

# "Retrieve all" query: an empty filter matches every document.
# The projection drops _id, because Dash can't turn an ObjectId into JSON.
df = pd.DataFrame.from_records(db.read({}, {"_id": 0}))

# Load the logo once and keep it as a base64 string for the layout
LOGO_FILE = "Grazioso Salvare Logo.png"
with open(LOGO_FILE, "rb") as logo:
    encoded_logo = base64.b64encode(logo.read()).decode()

# Quick checks before building anything on top of the data
print(len(df), "records loaded from aac.animals")
print("Animal types:", df["animal_type"].value_counts().to_dict())
print("Logo encoded:", len(encoded_logo), "characters")

10000 records loaded from aac.animals
Animal types: {'Dog': 5589, 'Cat': 3784, 'Other': 585, 'Bird': 40, 'Livestock': 2}
Logo encoded: 133148 characters


## Rescue profiles and queries
The profiles come from the Rescue Type and Preferred Dog Breeds table in the Dashboard Specifications Document. Each filter returns dogs of a preferred breed, of the right sex, within the right age range (in weeks). Reset sends an empty filter, which is the same "retrieve all" query used to build the unfiltered table.

| Rescue type | Preferred breeds | Sex | Age (weeks) |
|---|---|---|---|
| Water | Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland | Intact Female | 26 to 156 |
| Mountain or Wilderness | German Shepherd, Alaskan Malamute, Old English Sheepdog, Siberian Husky, Rottweiler | Intact Male | 26 to 156 |
| Disaster or Individual Tracking | Doberman Pinscher, German Shepherd, Golden Retriever, Bloodhound, Rottweiler | Intact Male | 20 to 300 |

In [4]:
###########################
# Model: rescue profile queries
###########################
# One entry per rescue type. The keys ("water", etc.) become the
# radio button values, so the controller can look up a profile directly.
RESCUE_PROFILES = {
    "water": {
        "label": "Water Rescue",
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever",
                   "Newfoundland"],
        "sex": "Intact Female",
        "min_weeks": 26,
        "max_weeks": 156,
    },
    "mountain": {
        "label": "Mountain or Wilderness Rescue",
        "breeds": ["German Shepherd", "Alaskan Malamute",
                   "Old English Sheepdog", "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 26,
        "max_weeks": 156,
    },
    "disaster": {
        "label": "Disaster or Individual Tracking",
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                   "Bloodhound", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 20,
        "max_weeks": 300,
    },
}


def build_query(filter_key):
    """Turn a radio button value into a MongoDB filter.

    Anything that isn't a known rescue type (including "reset")
    gives an empty filter, which returns every record.
    """
    profile = RESCUE_PROFILES.get(filter_key)
    if profile is None:
        return {}

    return {
        "animal_type": "Dog",
        # $in matches any one of the preferred breeds
        "breed": {"$in": profile["breeds"]},
        "sex_upon_outcome": profile["sex"],
        # $gte / $lte keep the age inside the training window
        "age_upon_outcome_in_weeks": {"$gte": profile["min_weeks"],
                                      "$lte": profile["max_weeks"]},
    }


def fetch_animals(filter_key):
    """Run the query through the CRUD module and return a DataFrame."""
    records = db.read(build_query(filter_key), {"_id": 0})
    # Passing columns keeps the table headers even when a query
    # matches nothing, so the table never collapses to a blank box
    return pd.DataFrame.from_records(records, columns=df.columns)


# Try every filter once before any widget depends on it
for key in list(RESCUE_PROFILES) + ["reset"]:
    label = RESCUE_PROFILES[key]["label"] if key in RESCUE_PROFILES else "Reset"
    result = fetch_animals(key)
    print(f"{label:<34} {len(result):>6} records")

Water Rescue                           17 records
Mountain or Wilderness Rescue           5 records
Disaster or Individual Tracking         4 records
Reset                               10000 records


## View: dashboard layout
The header carries the Grazioso Salvare logo and the developer identifier, so both appear in every screenshot. Below it, one row of radio buttons drives the whole page. Radio buttons fit here because the four choices are mutually exclusive and all visible at once. A user can't pick two rescue types by accident, and nothing has to be typed. The data table keeps the Module Six features (paging, sorting, column filters, single-row selection). Under the table, the breed pie chart and the location map sit side by side.

In [5]:
#########################
# View: dashboard layout
#########################
app = JupyterDash(__name__)

# Rows per page in the data table
PAGE_SIZE = 10

# Radio button choices, built from the profiles so labels never drift
# from the queries. Reset is last and is selected when the page opens.
FILTER_OPTIONS = [{"label": profile["label"], "value": key}
                  for key, profile in RESCUE_PROFILES.items()]
FILTER_OPTIONS.append({"label": "Reset (show all animals)", "value": "reset"})

app.layout = html.Div(style={"fontFamily": "Arial, sans-serif",
                             "margin": "0 20px"}, children=[

    # Header: client logo, title, and unique identifier
    html.Div(style={"display": "flex", "alignItems": "center", "gap": "24px"},
             children=[
        html.Img(src="data:image/png;base64," + encoded_logo,
                 alt="Grazioso Salvare logo",
                 style={"height": "110px"}),
        html.Div([
            html.H1("Grazioso Salvare Rescue Dog Dashboard",
                    style={"margin": "0"}),
            html.H4("Built by Sean Singh | CS 340 Project Two",
                    style={"margin": "6px 0 0 0", "color": "#555555"}),
        ]),
    ]),
    html.Hr(),

    # Controller input: one rescue type at a time
    html.Div([
        html.Label("Filter by rescue type:",
                   style={"fontWeight": "bold", "marginRight": "12px"}),
        dcc.RadioItems(
            id="filter-type",
            options=FILTER_OPTIONS,
            value="reset",
            labelStyle={"display": "inline-block", "marginRight": "24px"},
            inputStyle={"marginRight": "6px"},
        ),
        # Filled in by a callback, e.g. "Showing 17 records for Water Rescue"
        html.Div(id="match-count",
                 style={"marginTop": "8px", "fontStyle": "italic"}),
    ]),
    html.Hr(),

    # Interactive data table, starting with the unfiltered data
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": col, "id": col, "deletable": False,
                  "selectable": True} for col in df.columns],
        data=df.to_dict("records"),
        page_action="native",        # paging keeps 10,000 rows usable
        page_size=PAGE_SIZE,
        sort_action="native",        # click a header to sort
        sort_mode="multi",           # shift-click to sort by several columns
        filter_action="native",      # text filter under each header
        row_selectable="single",     # the selected row drives the map
        selected_rows=[0],
        column_selectable="single",  # highlight a column
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "90px",
                    "maxWidth": "220px", "whiteSpace": "normal"},
        style_header={"fontWeight": "bold", "backgroundColor": "#EEEEEE"},
    ),

    html.Br(),
    html.Hr(),

    # Charts: breed pie chart on the left, location map on the right
    html.Div(style={"display": "flex", "gap": "20px"}, children=[
        html.Div(id="graph-id", style={"flex": "1", "minWidth": "0"}),
        html.Div(id="map-id", style={"flex": "1", "minWidth": "0"}),
    ]),
])

## Controller: callbacks
`update_dashboard` is the only callback that talks to MongoDB. It turns the radio button choice into a query with `fetch_animals()` and sends the results to the table. The charts then follow the table: the pie chart groups the visible rows by breed, and the map marks whichever row is selected. Because they read the table's *derived* data (the rows left after any sorting or column filtering), the charts always match what the user is looking at.

In [6]:
#############################################
# Controller: interaction between components
#############################################
# Number of breeds drawn as their own pie slices; the rest become "Other"
TOP_BREEDS = 8


@app.callback(
    [Output("datatable-id", "data"),
     Output("datatable-id", "selected_rows"),
     Output("datatable-id", "page_current"),
     Output("datatable-id", "filter_query"),
     Output("datatable-id", "sort_by"),
     Output("match-count", "children")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Query MongoDB for the chosen rescue type and refresh the table."""
    dff = fetch_animals(filter_type)

    if filter_type in RESCUE_PROFILES:
        label = RESCUE_PROFILES[filter_type]["label"]
    else:
        label = "all animals (no filter)"
    message = f"Showing {len(dff):,} records for {label}."

    # Select the first row so the map has something to show, go back
    # to page one, and clear any leftover column filters or sorting.
    # That way Reset truly returns the table to its starting state.
    first_row = [0] if len(dff) > 0 else []
    return dff.to_dict("records"), first_row, 0, "", [], message


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Shade every cell in the column the user selects."""
    if not selected_columns:
        return []
    return [{"if": {"column_id": col}, "background_color": "#D2F3FF"}
            for col in selected_columns]


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):
    """Pie chart of the breeds currently shown in the table."""
    dff = df if view_data is None else pd.DataFrame.from_dict(view_data)
    if dff.empty:
        return [html.P("No records match the current filter.")]

    # The unfiltered data has hundreds of breeds, so keep the most
    # common ones and fold the long tail into a single "Other" slice
    counts = dff["breed"].value_counts()
    pie_df = pd.DataFrame({"breed": counts.index[:TOP_BREEDS],
                           "count": counts.values[:TOP_BREEDS]})
    other = counts.values[TOP_BREEDS:].sum()
    if other > 0:
        pie_df = pd.concat([pie_df, pd.DataFrame({"breed": ["Other"],
                                                  "count": [other]})],
                           ignore_index=True)

    fig = px.pie(pie_df, names="breed", values="count",
                 title=f"Breed mix of the {len(dff):,} animals in the table")
    fig.update_traces(textposition="inside", textinfo="percent+label")
    fig.update_layout(height=500, margin={"t": 60, "b": 20, "l": 20, "r": 20})
    return [dcc.Graph(figure=fig)]


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(view_data, selected):
    """Mark the selected animal's location on a map of the Austin area."""
    dff = df if view_data is None else pd.DataFrame.from_dict(view_data)
    if dff.empty:
        return [html.P("No records match the current filter.")]

    # Single-row selection means the list holds at most one index
    row = selected[0] if selected else 0
    if row >= len(dff):
        row = 0

    record = dff.iloc[row]
    lat, lon = record["location_lat"], record["location_long"]
    if pd.isna(lat) or pd.isna(lon):
        return [html.P("The selected animal has no location on record.")]

    name = record["name"] if pd.notna(record["name"]) and record["name"] \
        else "(no name on record)"

    return [
        html.H4("Location of the selected animal",
                style={"textAlign": "center", "margin": "0 0 8px 0"}),
        dl.Map(style={"width": "100%", "height": "460px"},
               center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Hover shows the breed; clicking shows the name
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(record["breed"]),
                dl.Popup([html.H3("Animal Name"), html.P(name)]),
            ]),
        ]),
    ]

In [7]:
# Start the dashboard server
app.run_server(debug=True)

Dash app running on https://salmonfrank-paulzero-3000.codio.io/proxy/8050/
